In [55]:
import re
import warnings
import unicodedata
from collections import Counter
import math

import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")



MAX_TOKEN_LEN = 30
MAX_PAIRS = 20_000
NHEAD = 4
NUM_LAYERS = 2
D_MODEL = 128
D_FF = 512

In [56]:
df = pd.read_csv("english_french.csv")

df = df.dropna(subset=["English", "French"]).drop_duplicates().reset_index(drop=True)

df.head()

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [57]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower().strip()

    text = re.sub(r"([.!?,;:])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)

    return text


def tokenize(text):
    return clean_text(text).split()


df["en_tokens"] = df["English"].map(tokenize)
df["fr_tokens"] = df["French"].map(tokenize)

In [58]:
train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train, val = train_test_split(
    train,
    test_size=0.1,
    random_state=42
)

print(f"Training samples:   {len(train):,}")
print(f"Validation samples: {len(val):,}")
print(f"Testing samples:    {len(test):,}")

Training samples:   165,457
Validation samples: 18,385
Testing samples:    45,961


In [59]:
def build_vocab(tokenized_texts):
    counter = Counter()

    for tokens in tokenized_texts:
        counter.update(tokens)

    vocab = {
        "<pad>": 0,
        "<unk>": 1,
        "<bos>": 2,
        "<eos>": 3
    }

    for token, _ in counter.most_common():
        if token not in vocab:
            vocab[token] = len(vocab)

    return vocab


src_vocab = build_vocab(train["en_tokens"])
tgt_vocab = build_vocab(train["fr_tokens"])

print(f"Source vocabulary size: {len(src_vocab):,}")
print(f"Target vocabulary size: {len(tgt_vocab):,}")

Source vocabulary size: 15,593
Target vocabulary size: 29,461


In [60]:
def encode_tokens(tokens, vocab):
    token_ids = [vocab["<bos>"]]

    token_ids.extend(
        vocab.get(token, vocab["<unk>"])
        for token in tokens
    )

    token_ids.append(vocab["<eos>"])

    if len(token_ids) > MAX_TOKEN_LEN:
        token_ids = token_ids[:MAX_TOKEN_LEN - 1]
        token_ids.append(vocab["<eos>"])

    return token_ids


def pad_sequences(token_ids, vocab):
    padding_length = MAX_TOKEN_LEN - len(token_ids)

    if padding_length > 0:
        token_ids = token_ids + [vocab["<pad>"]] * padding_length

    return token_ids

In [61]:
def prepare_data(data, src_vocab, tgt_vocab):
    source_sequences = []
    target_sequences = []

    for en_tokens, fr_tokens in zip(
        data["en_tokens"], data["fr_tokens"]
    ):
        source_ids = encode_tokens(en_tokens, src_vocab)
        target_ids = encode_tokens(fr_tokens, tgt_vocab)

        source_sequences.append(
            pad_sequences(source_ids, src_vocab)
        )
        target_sequences.append(
            pad_sequences(target_ids, tgt_vocab)
        )

    return (
        torch.tensor(source_sequences, dtype=torch.long),
        torch.tensor(target_sequences, dtype=torch.long)
    )


X_train, Y_train = prepare_data(train, src_vocab, tgt_vocab)
X_val, Y_val = prepare_data(val, src_vocab, tgt_vocab)
X_test, Y_test = prepare_data(test, src_vocab, tgt_vocab)

print("Training:", X_train.shape, Y_train.shape)
print("Validation:", X_val.shape, Y_val.shape)
print("Testing:", X_test.shape, Y_test.shape)

Training: torch.Size([165457, 30]) torch.Size([165457, 30])
Validation: torch.Size([18385, 30]) torch.Size([18385, 30])
Testing: torch.Size([45961, 30]) torch.Size([45961, 30])


In [62]:
BATCH_SIZE = 32

train_loader = DataLoader(
    TensorDataset(X_train, Y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, Y_val),
    batch_size=BATCH_SIZE
)

test_loader = DataLoader(
    TensorDataset(X_test, Y_test),
    batch_size=BATCH_SIZE
)

In [63]:
class EncoderDecoderTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, nhead, num_layers, d_ff, max_length, dropout, src_pad_id, tgt_pad_id):
        super().__init__()

        self.src_pad_id = src_pad_id
        self.tgt_pad_id = tgt_pad_id
        self.d_model = d_model

        self.src_embedding = nn.Embedding(
            src_vocab_size,
            d_model,
            padding_idx=src_pad_id
        )

        self.tgt_embedding = nn.Embedding(
            tgt_vocab_size,
            d_model,
            padding_idx=tgt_pad_id
        )

        self.position_embedding = nn.Embedding(
            max_length,
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        self.output_layer = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    def forward(self, source, target_input):
        source_length = source.size(1)
        target_length = target_input.size(1)

        source_positions = torch.arange(source_length, device=source.device)
        target_positions = torch.arange(target_length, device=target_input.device)

        source_embeddings = self.src_embedding(source) * math.sqrt(self.d_model)
        target_embeddings = self.tgt_embedding(target_input) * math.sqrt(self.d_model)

        source_embeddings += self.position_embedding(source_positions)
        target_embeddings += self.position_embedding(target_positions)

        source_padding_mask = source.eq(self.src_pad_id)
        target_padding_mask = target_input.eq(self.tgt_pad_id)

        memory = self.encoder(
            source_embeddings,
            src_key_padding_mask=source_padding_mask
        )

        causal_mask = torch.triu(
            torch.ones(
                target_length,
                target_length,
                dtype=torch.bool,
                device=target_input.device
            ),
            diagonal=1
        )

        decoder_output = self.decoder(
            tgt=target_embeddings,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask
        )

        return self.output_layer(decoder_output)

In [64]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
model = EncoderDecoderTransformer(
    len(src_vocab), len(tgt_vocab),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_length=MAX_TOKEN_LEN,
    dropout=0.3,
    src_pad_id=src_vocab["<pad>"],
    tgt_pad_id=tgt_vocab["<pad>"]
).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 10,496,917


In [65]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.01)

In [66]:
EPOCHS = 10


def run_epoch(loader, training):
    model.train(training)

    total_loss = 0.0
    total_tokens = 0

    for source, target in loader:
        source = source.to(device)
        target = target.to(device)

        target_input = target[:, :-1]
        target_output = target[:, 1:]

        with torch.set_grad_enabled(training):
            logits = model(source, target_input)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                target_output.reshape(-1)
            )

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0
                )

                optimizer.step()

        valid_tokens = target_output.ne(
            tgt_vocab["<pad>"]
        ).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


best_val_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, training=True)

    with torch.no_grad():
        val_loss = run_epoch(val_loader, training=False)

    print(f"Epoch {epoch + 1}/{EPOCHS} | "f"Train Loss: {train_loss:.4f} | "f"Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "english_french_transformer.pth")

Epoch 1/10 | Train Loss: 1.1796 | Val Loss: 0.8487
Epoch 2/10 | Train Loss: 0.8091 | Val Loss: 0.6889
Epoch 3/10 | Train Loss: 0.6823 | Val Loss: 0.6148
Epoch 4/10 | Train Loss: 0.6051 | Val Loss: 0.5641
Epoch 5/10 | Train Loss: 0.5514 | Val Loss: 0.5336
Epoch 6/10 | Train Loss: 0.5119 | Val Loss: 0.5150
Epoch 7/10 | Train Loss: 0.4807 | Val Loss: 0.4951
Epoch 8/10 | Train Loss: 0.4559 | Val Loss: 0.4842
Epoch 9/10 | Train Loss: 0.4350 | Val Loss: 0.4751
Epoch 10/10 | Train Loss: 0.4184 | Val Loss: 0.4654


In [67]:
model.load_state_dict(
    torch.load(
        "english_french_transformer.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

with torch.no_grad():
    test_loss = run_epoch(test_loader, training=False)

print(f"Test Loss: {test_loss:.4f}")

Test Loss: 0.4606


In [68]:
id_to_tgt = {
    idx: token for token, idx in tgt_vocab.items()
}


def translate(sentence):
    model.eval()

    tokens = tokenize(sentence)
    source_ids = encode_tokens(tokens, src_vocab)
    source_ids = pad_sequences(source_ids, src_vocab)

    source = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=device
    )

    generated = [tgt_vocab["<bos>"]]

    with torch.no_grad():
        for _ in range(MAX_TOKEN_LEN - 1):
            target_input = torch.tensor(
                [generated],
                dtype=torch.long,
                device=device
            )

            logits = model(source, target_input)
            next_token_id = logits[0, -1].argmax().item()

            if next_token_id == tgt_vocab["<eos>"]:
                break

            generated.append(next_token_id)

    output_tokens = [
        id_to_tgt[token_id]
        for token_id in generated[1:]
        if token_id not in (
            tgt_vocab["<pad>"],
            tgt_vocab["<bos>"],
            tgt_vocab["<eos>"]
        )
    ]

    return " ".join(output_tokens)


print(translate("I love machine learning."))
print(translate("How are you?"))
print(translate("Good morning!"))

j'adore les poches .
comment es-tu ?
bon matin !
